# Experiment: one agentic Gemini call → story map → code picks the cut

This notebook tests the new idea **before** we change the pipeline. It's standalone: it doesn't need the fovea repo.

1. **Units (code):** the video is cut into small exact pieces: phrases on word boundaries (Whisper) and
   stretches without speech, never across a shot cut (PySceneDetect).
2. **Story map (Gemini, ONE call):** Gemini watches the whole video with **agentic video understanding**
   and scores / links the units by id. There's no output schema, so we ask for one ```json block and parse it.
3. **Selector (code):** picks units for each target duration (20 s, 15 s, 10 s, …). It's instant and needs no
   more Gemini calls, so re-run the tuning cell as often as you like.
4. **Preview:** quick 16:9 cuts (no reframing) for `synced` and `cutaway`.

Runtime: a GPU makes Whisper faster, but CPU works for short clips. Add a Colab secret **`GEMINI_API_KEY`** (🔑 icon).

## 1. Install

In [ ]:
!pip install -q -U google-genai faster-whisper "scenedetect[opencv]"
from google.genai import types
assert hasattr(types, "MediaProcessing"), "this google-genai version has no agentic video support - upgrade it"
print("agentic video supported:", [m.value for m in types.MediaProcessing])

## 2. Settings

In [ ]:
MODEL = "gemini-flash-latest"  # @param ["gemini-flash-latest", "gemini-pro-latest", "gemini-3.8-flash", "gemini-3.5-flash", "gemini-2.5-pro", "gemini-2.5-flash"] {allow-input: true}
PROCESSING = "AGENTIC"  # @param ["AGENTIC", "STATIC"]
TARGETS = "20, 15, 10"  # @param {type:"string"}
MODES = ["synced", "cutaway"]
WHISPER_MODEL = "small"  # @param ["tiny", "base", "small", "medium"]
SHOT_THRESHOLD = 27.0  # @param {type:"number"}
# Optional: a fovea job folder (e.g. on Drive: /content/drive/MyDrive/fovea_data/users/colab-test/<job_id>).
# Then shots, words and TalkNet's "who is visibly speaking" come from the real pipeline.
JOB_DIR = ""  # @param {type:"string"}

## 3. Gemini key

In [ ]:
import getpass
try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_KEY = None
GEMINI_KEY = GEMINI_KEY or getpass.getpass("Gemini API key: ")
from google import genai
client = genai.Client(api_key=GEMINI_KEY)
print("ok")

## 4. Video

In [ ]:
import os, glob
if JOB_DIR:
    VIDEO = sorted(glob.glob(os.path.join(JOB_DIR, "input", "*")))[0]
else:
    from google.colab import files
    up = files.upload()
    VIDEO = "/content/" + list(up)[0]
print(VIDEO)
from IPython.display import Video, display
display(Video(VIDEO, embed=True, width=480))

## 5. Shots, words (and who is visibly speaking)

In [ ]:
import json, pickle, time
import pandas as pd

SPEAKING_AT = None
if JOB_DIR:
    sdf = pd.read_csv(os.path.join(JOB_DIR, "prep", "shots.csv")).sort_values("start_sec")
    SHOTS = list(zip(sdf.start_sec.astype(float), sdf.end_sec.astype(float)))
    WORDS = []
    tr = os.path.join(JOB_DIR, "audio", "transcript.json")
    for seg in (json.load(open(tr)) if os.path.exists(tr) else []):
        for w in seg.get("words_data") or []:
            WORDS.append({"start": float(w["start"]), "end": float(w["end"]), "word": w["word"]})
    info = json.load(open(os.path.join(JOB_DIR, "prep", "video_info.json")))
    fps = info["proxy"]["fps"]
    speaking = set()
    for pkl in sorted(glob.glob(os.path.join(JOB_DIR, "visual", "shard_*.pkl"))):
        for trk in pickle.load(open(pkl, "rb"))["speaker_scores"]:
            speaking.update(int(f) for f, s in zip(trk["frames"], trk["scores"]) if float(s) > 0)
    def SPEAKING_AT(a, b):
        fr = range(int(a * fps), max(int(a * fps) + 1, int(b * fps)))
        return sum(f in speaking for f in fr) / len(fr)
    if not WORDS:
        print("no transcript in the job folder (was transcribe on?) - running Whisper")
if not JOB_DIR:
    from scenedetect import detect, ContentDetector
    t = time.time()
    scenes = detect(VIDEO, ContentDetector(threshold=SHOT_THRESHOLD))
    SHOTS = [(s.get_seconds(), e.get_seconds()) for s, e in scenes]
    print(f"{len(SHOTS)} shots ({time.time() - t:.1f}s)")
if not SHOTS:
    import cv2
    cap = cv2.VideoCapture(VIDEO)
    SHOTS = [(0.0, cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS))]
if not JOB_DIR or not WORDS:
    from faster_whisper import WhisperModel
    t = time.time()
    try:
        wm = WhisperModel(WHISPER_MODEL, device="cuda", compute_type="float16")
    except Exception:
        wm = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")
    segs, _ = wm.transcribe(VIDEO, word_timestamps=True, vad_filter=True)
    WORDS = [{"start": w.start, "end": w.end, "word": w.word} for s in segs for w in (s.words or [])]
    print(f"{len(WORDS)} words ({time.time() - t:.1f}s)")
print(" ".join(w["word"].strip() for w in WORDS)[:1000])

## 6. Experiment code (units, prompt, parser, selector, preview)
The same code was tested offline; run this cell as is.

In [ ]:
# ==== story-map experiment library (pasted into the notebook as one cell; tested locally) ====
import json
import math
import re
import subprocess
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Tuple

# ----------------------------------------------------------------------------- units
@dataclass
class Unit:
    id: str
    start: float
    end: float
    shot: int
    kind: str                 # "speech" | "picture"
    text: str = ""
    speaking: Optional[float] = None   # share of frames with a visibly speaking person (None = unknown)

    @property
    def dur(self) -> float:
        return self.end - self.start


def build_units(shots: List[Tuple[float, float]], words: List[Dict[str, Any]],
                speaking_at=None, max_gap: float = 0.35, max_speech: float = 6.0,
                min_picture: float = 0.5, max_picture: float = 2.5) -> List[Unit]:
    """Exact, small pieces of the video. Speech units = phrases on word boundaries (never across a
    shot cut); picture units = stretches without speech, split into <= max_picture chunks."""
    def shot_of(t):
        for i, (a, b) in enumerate(shots):
            if a <= t < b:
                return i
        return len(shots) - 1

    words = sorted((w for w in words if w["end"] > w["start"]), key=lambda w: w["start"])
    phrases: List[List[Dict[str, Any]]] = []
    for w in words:
        if phrases:
            cur = phrases[-1]
            prev = cur[-1]
            gap = w["start"] - prev["end"]
            dur = prev["end"] - cur[0]["start"]
            new = (gap > max_gap
                   or re.search(r"[.?!]$", prev["word"].strip()) is not None
                   or (re.search(r"[,;:]$", prev["word"].strip()) and dur > 2.5)
                   or (dur > max_speech and gap > 0.12)
                   or shot_of((w["start"] + w["end"]) / 2) != shot_of((prev["start"] + prev["end"]) / 2))
            if not new:
                cur.append(w)
                continue
        phrases.append([w])

    raw: List[Tuple[float, float, int, str, str]] = []
    for k, ph in enumerate(phrases):
        s = shot_of((ph[0]["start"] + ph[0]["end"]) / 2)
        a0, b0 = shots[s]
        prev_end = phrases[k - 1][-1]["end"] if k else 0.0
        next_start = phrases[k + 1][0]["start"] if k + 1 < len(phrases) else 1e9
        start = max(a0, ph[0]["start"] - 0.08, (prev_end + ph[0]["start"]) / 2 if k else 0.0)
        end = min(b0, ph[-1]["end"] + 0.12, (ph[-1]["end"] + next_start) / 2)
        raw.append((start, end, s, "speech", " ".join(w["word"].strip() for w in ph)))

    # picture units: uncovered stretches inside each shot
    for s, (a, b) in enumerate(shots):
        covered = sorted((r[0], r[1]) for r in raw if r[2] == s and r[3] == "speech")
        t = a
        holes = []
        for c0, c1 in covered + [(b, b)]:
            if c0 - t >= min_picture:
                holes.append((t, c0))
            t = max(t, c1)
        for h0, h1 in holes:
            n = max(1, math.ceil((h1 - h0) / max_picture))
            step = (h1 - h0) / n
            for i in range(n):
                raw.append((h0 + i * step, h0 + (i + 1) * step, s, "picture", ""))

    raw.sort()
    units = []
    for i, (a, b, s, kind, text) in enumerate(raw):
        units.append(Unit(f"U{i + 1:02d}", round(a, 3), round(b, 3), s, kind, text,
                          speaking_at(a, b) if speaking_at else None))
    return units


def units_table(units: List[Unit]) -> str:
    lines = []
    for u in units:
        sp = "unknown" if u.speaking is None else f"{u.speaking:.0%}"
        txt = f'"{u.text}"' if u.text else "(no speech)"
        lines.append(f"{u.id} | {u.start:.2f}-{u.end:.2f}s ({u.dur:.2f}s) | shot {u.shot} | {u.kind} | "
                     f"visible speaker: {sp} | {txt}")
    return "\n".join(lines)


# ----------------------------------------------------------------------------- prompt
STORY_PROMPT = """You are a senior video editor. Watch this video (rewatch any part as often as you need)
and build a STORY MAP that a program will use to cut shorter versions of it (e.g. 20 s, 15 s, 10 s).

The video has been cut into UNITS with exact times (below). Refer to units ONLY by their id (U01, U02, ...);
never invent timestamps except in "moments". A "speech" unit is a phrase someone says; a "picture" unit is a
stretch without speech.

For every unit decide:
  importance 0-10  how much the short version needs it to tell the story (0 = pure filler / dead time)
  kind             one of: dialogue (someone in the scene talks), voiceover (narration / off-screen speech),
                   reaction (a visible reaction, no talking), visual (an action, object or scene worth seeing),
                   filler (nothing worth keeping)
  shows            what is SEEN in this unit (short)
  note             why it matters (or why it doesn't)

Also give:
  needs     units that only make sense together (a punchline needs its setup; an answer needs its question)
  cutaways  for speech units: picture or reaction units that could be shown INSTEAD of the speaker while the line
            plays (e.g. show the gift while she names it). Never propose a unit where someone is visibly talking.
  hook      the most gripping unit(s) to open a short version with, if moving them first helps
  moments   optional exact peaks (seconds in the source video) inside a unit, e.g. the frame the reaction lands

Answer with your reasoning first if you like, then ONE ```json block exactly in this shape:
```json
{
  "summary": "one or two sentences: the story of the video",
  "units": [{"id": "U01", "importance": 0, "kind": "filler", "shows": "...", "note": "..."}],
  "needs": [{"unit": "U07", "requires": ["U03"], "why": "..."}],
  "cutaways": [{"speech": "U07", "pictures": ["U11"], "fit": 9, "why": "..."}],
  "hook": {"units": ["U08"], "why": "..."},
  "moments": [{"unit": "U08", "peak_s": 21.4, "note": "..."}]
}
```
Include EVERY unit in "units".

UNITS:
{units}
"""


def story_prompt(units: List[Unit]) -> str:
    return STORY_PROMPT.replace("{units}", units_table(units))


# ----------------------------------------------------------------------------- parsing
def extract_json(text: str) -> Optional[Dict[str, Any]]:
    """Last ```json block; else the last {...} that parses."""
    blocks = re.findall(r"```(?:json)?\s*(\{.*?\})\s*```", text or "", flags=re.S)
    for b in reversed(blocks):
        try:
            return json.loads(b)
        except json.JSONDecodeError:
            continue
    start = (text or "").rfind("{\n")
    while start != -1:
        try:
            return json.loads(text[start:])
        except json.JSONDecodeError:
            start = text.rfind("{\n", 0, start)
    return None


def clean_story_map(raw: Dict[str, Any], units: List[Unit]) -> Tuple[Dict[str, Any], List[str]]:
    """Keep only known unit ids, clamp numbers, fill units Gemini forgot (importance 0)."""
    ids = {u.id for u in units}
    issues = []
    by_id = {}
    for u in raw.get("units", []) or []:
        uid = str(u.get("id", "")).strip()
        if uid not in ids:
            issues.append(f"unknown unit {uid!r} ignored")
            continue
        try:
            imp = float(u.get("importance", 0))
        except (TypeError, ValueError):
            imp = 0.0
        by_id[uid] = {"id": uid, "importance": max(0.0, min(10.0, imp)),
                      "kind": str(u.get("kind", "filler")), "shows": str(u.get("shows", "")),
                      "note": str(u.get("note", ""))}
    for u in units:
        if u.id not in by_id:
            issues.append(f"{u.id} missing from the answer -> importance 0")
            by_id[u.id] = {"id": u.id, "importance": 0.0, "kind": "filler", "shows": "", "note": "(missing)"}
    needs = [{"unit": n["unit"], "requires": [r for r in n.get("requires", []) if r in ids], "why": n.get("why", "")}
             for n in raw.get("needs", []) or [] if n.get("unit") in ids]
    cutaways = []
    for c in raw.get("cutaways", []) or []:
        if c.get("speech") in ids:
            pics = [p for p in c.get("pictures", []) if p in ids]
            if pics:
                cutaways.append({"speech": c["speech"], "pictures": pics, "fit": float(c.get("fit", 5)),
                                 "why": c.get("why", "")})
    hook = raw.get("hook") or {}
    hook = {"units": [h for h in hook.get("units", []) if h in ids], "why": hook.get("why", "")}
    moments = [m for m in raw.get("moments", []) or [] if m.get("unit") in ids]
    return {"summary": raw.get("summary", ""), "units": [by_id[u.id] for u in units], "needs": needs,
            "cutaways": cutaways, "hook": hook, "moments": moments}, issues


# ----------------------------------------------------------------------------- selection
@dataclass
class Piece:
    """One output piece: `length` seconds of picture from video_start and sound from audio_start."""
    video_start: float
    audio_start: float
    length: float
    units: str
    note: str = ""


def unit_value(importance: float, dur: float, imp_power: float = 1.5, dur_power: float = 0.5) -> float:
    """How much a unit is worth keeping: importance matters most, length a little (so a short punchline
    beats a long so-so stretch)."""
    return (max(0.0, importance) ** imp_power) * (max(dur, 0.05) ** dur_power)


def select_units(units: List[Unit], smap: Dict[str, Any], target: float, mode: str = "synced",
                 cut_penalty: float = 3.0, tolerance: float = 0.1, use_hook: bool = True,
                 min_cutaway: float = 0.8, imp_power: float = 1.5, dur_power: float = 0.5,
                 min_importance: float = 1.0) -> Tuple[List[Piece], Dict[str, Any]]:
    """Pick units to fit `target` seconds (<= target, >= (1-tolerance)*target if possible).

    * `needs` links glue units into BUNDLES that are kept or dropped together (setup + punchline)
    * 0/1 knapsack over bundles: value = sum of unit_value, minus `cut_penalty` per separate run
    * in cutaway mode, picture units that can cover a chosen line are shown OVER it instead of
      taking their own time
    """
    info = {m["id"]: m for m in smap["units"]}
    by_id = {u.id: u for u in units}
    pos = {u.id: i for i, u in enumerate(units)}
    imp = {u.id: info[u.id]["importance"] for u in units}

    overlay_ok: Dict[str, List[Tuple[float, str]]] = {}   # speech unit -> usable overlay pictures
    for c in smap["cutaways"]:
        for p in c["pictures"]:
            pu = by_id[p]
            if (pu.speaking or 0) < 0.2 and info[p]["kind"] not in ("dialogue", "voiceover"):
                overlay_ok.setdefault(c["speech"], []).append((c["fit"], p))
    overlays = {p for lst in overlay_ok.values() for _, p in lst}

    # bundles = connected groups of `needs` links (union-find)
    parent = {u.id: u.id for u in units}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for n in smap["needs"]:
        for r in n["requires"]:
            parent[find(n["unit"])] = find(r)
    groups: Dict[str, List[str]] = {}
    for u in units:
        if imp[u.id] < min_importance and find(u.id) == u.id:
            continue   # plain filler is never picked
        if mode == "cutaway" and u.id in overlays and find(u.id) == u.id:
            continue   # used as an overlay instead
        groups.setdefault(find(u.id), []).append(u.id)

    def runs(ids):
        idx = sorted(pos[i] for i in ids)
        return 1 + sum(1 for a, b in zip(idx, idx[1:]) if b != a + 1)

    step = 0.1
    cap = int(round(target / step))
    items = []
    for ids in groups.values():
        w = max(1, int(round(sum(by_id[i].dur for i in ids) / step)))
        v = sum(unit_value(imp[i], by_id[i].dur, imp_power, dur_power) for i in ids) - cut_penalty * runs(ids)
        if w <= cap and v > 0:
            items.append((w, v, ids))

    # knapsack
    best = [0.0] + [-1e18] * cap
    keep = [[] for _ in range(cap + 1)]
    for w, v, ids in items:
        for t in range(cap, w - 1, -1):
            if best[t - w] > -1e17 and best[t - w] + v > best[t]:
                best[t] = best[t - w] + v
                keep[t] = keep[t - w] + [ids]
    lo = int(math.floor((1 - tolerance) * cap))
    reachable = [t for t in range(cap + 1) if best[t] > -1e17]
    in_range = [t for t in reachable if t >= lo]
    t_best = max(in_range or reachable, key=lambda t: (best[t], t))
    chosen = {i for ids in keep[t_best] for i in ids}

    notes = []
    if not in_range:
        notes.append(f"could not reach {(1 - tolerance) * target:.1f}s with units worth keeping")
    order = [u for u in units if u.id in chosen]
    if use_hook:
        required = {r for n in smap["needs"] for r in n["requires"]}
        hook_ids = [h for h in smap["hook"]["units"] if h in chosen and h not in required
                    and not any(n["unit"] == h and n["requires"] for n in smap["needs"])]
        if hook_ids:
            order = [u for u in order if u.id in hook_ids] + [u for u in order if u.id not in hook_ids]
            notes.append(f"hook first: {', '.join(hook_ids)}")
        elif smap["hook"]["units"]:
            notes.append("hook not moved (not chosen, or it depends on a setup)")

    pieces: List[Piece] = []
    used = set()
    for u in order:
        if mode == "cutaway" and u.kind == "speech" and u.id in overlay_ok:
            options = [(f, p) for f, p in sorted(overlay_ok[u.id], reverse=True)
                       if p not in used and by_id[p].dur >= min_cutaway]
            if options:
                _, p = options[0]
                pu = by_id[p]
                used.add(p)
                cut_len = min(pu.dur, u.dur)
                own = u.dur - cut_len
                label = f"cutaway: {info[p]['shows']}"
                if own >= 0.3:   # speaker first, then cut to the picture while the line finishes
                    pieces.append(Piece(u.start, u.start, own, u.id, "speaker (synced)"))
                    pieces.append(Piece(pu.start, u.start + own, cut_len, f"{u.id}<-{p}", label))
                else:
                    pieces.append(Piece(pu.start, u.start, u.dur, f"{u.id}<-{p}", label))
                continue
        pieces.append(Piece(u.start, u.start, u.dur, u.id, info[u.id]["note"][:60]))
    stats = {"target": target, "mode": mode, "total": round(sum(p.length for p in pieces), 2),
             "units": [u.id for u in order], "notes": notes}
    return pieces, stats


# ----------------------------------------------------------------------------- preview render
def render_preview(src: str, pieces: List[Piece], out: str, width: int = 640) -> str:
    """Quick 16:9 preview (no reframing): every piece = picture from video_start + sound from audio_start."""
    import os
    import tempfile
    tmp = tempfile.mkdtemp()
    parts = []
    for i, p in enumerate(pieces):
        part = os.path.join(tmp, f"p{i:03d}.mp4")
        subprocess.run(["ffmpeg", "-v", "error", "-y",
                        "-ss", f"{p.video_start:.3f}", "-t", f"{p.length:.3f}", "-i", src,
                        "-ss", f"{p.audio_start:.3f}", "-t", f"{p.length:.3f}", "-i", src,
                        "-map", "0:v:0", "-map", "1:a:0?", "-vf", f"scale={width}:-2,fps=25",
                        "-af", "afade=t=in:d=0.015,areverse,afade=t=in:d=0.015,areverse",
                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-pix_fmt", "yuv420p",
                        "-c:a", "aac", "-ar", "48000", "-ac", "2", part], check=True)
        parts.append(part)
    lst = os.path.join(tmp, "list.txt")
    with open(lst, "w") as f:
        f.writelines(f"file '{p}'\n" for p in parts)
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-f", "concat", "-safe", "0", "-i", lst,
                    "-c:v", "libx264", "-preset", "veryfast", "-crf", "23", "-c:a", "aac", out], check=True)
    return out

## 7. Units

In [ ]:
UNITS = build_units(SHOTS, WORDS, SPEAKING_AT)
print(f"{len(UNITS)} units ({sum(u.kind == 'speech' for u in UNITS)} speech, {sum(u.kind == 'picture' for u in UNITS)} picture)\n")
print(units_table(UNITS))

## 8. ONE agentic Gemini call → story map

In [ ]:
from google.genai import types

def upload(path):
    f = client.files.upload(file=path)
    while f.state and f.state.name == "PROCESSING":
        time.sleep(2)
        f = client.files.get(name=f.name)
    assert f.state.name == "ACTIVE", f
    return f

if "VIDEO_FILE" not in globals() or VIDEO_FILE_PATH != VIDEO:
    VIDEO_FILE, VIDEO_FILE_PATH = upload(VIDEO), VIDEO

def story_call(processing=PROCESSING, model=MODEL):
    video_part = types.Part(file_data=types.FileData(file_uri=VIDEO_FILE.uri, mime_type=VIDEO_FILE.mime_type),
                            media_processing=processing)
    t = time.time()
    resp = client.models.generate_content(model=model, contents=[video_part, story_prompt(UNITS)])
    secs = time.time() - t
    u = resp.usage_metadata
    print(f"{model} / {processing}: {secs:.1f}s | tokens in={getattr(u, 'prompt_token_count', None)} "
          f"out={getattr(u, 'candidates_token_count', None)} thinking={getattr(u, 'thoughts_token_count', None)}")
    return resp.text, secs

RAW_TEXT, SECS = story_call()
open(f"/content/story_map_raw_{PROCESSING}.txt", "w").write(RAW_TEXT)
print(RAW_TEXT[:6000])

## 9. Parse (lenient) → fallback: a cheap structured call that only reformats the answer

In [ ]:
from typing import List
from pydantic import BaseModel

class SUnit(BaseModel):
    id: str; importance: float; kind: str; shows: str; note: str
class SNeed(BaseModel):
    unit: str; requires: List[str]; why: str
class SCut(BaseModel):
    speech: str; pictures: List[str]; fit: float; why: str
class SHook(BaseModel):
    units: List[str]; why: str
class SMoment(BaseModel):
    unit: str; peak_s: float; note: str
class SMap(BaseModel):
    summary: str; units: List[SUnit]; needs: List[SNeed]; cutaways: List[SCut]; hook: SHook; moments: List[SMoment]

def parse_story(text):
    raw = extract_json(text)
    how = "parsed the ```json block"
    if raw is None:
        how = "fallback: reformatted by a structured call"
        fix = client.models.generate_content(
            model="gemini-flash-latest",
            contents=["Convert this video editor's answer into the JSON schema. Keep every value; invent nothing.\n\n" + text],
            config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=SMap))
        raw = json.loads(fix.text)
    smap, issues = clean_story_map(raw, UNITS)
    print(how, "|", len(issues), "issue(s)", issues[:10])
    return smap

SMAP = parse_story(RAW_TEXT)
print("\nSUMMARY:", SMAP["summary"])
print("HOOK:", SMAP["hook"])
df = pd.DataFrame([{**{"id": u.id, "start": u.start, "end": u.end, "kind_unit": u.kind, "text": u.text}, **m}
                   for u, m in zip(UNITS, SMAP["units"])])
pd.set_option("display.max_colwidth", 80)
display(df[["id", "start", "end", "importance", "kind", "text", "shows", "note"]])
print("NEEDS:", *SMAP["needs"], sep="\n  ")
print("CUTAWAYS:", *SMAP["cutaways"], sep="\n  ")
print("MOMENTS:", *SMAP["moments"], sep="\n  ")

## 10. Cuts for every target × mode (code only, instant) + previews

In [ ]:
TARGET_LIST = [float(t) for t in TARGETS.split(",") if t.strip()]
RESULTS = {}
def run_cuts(smap=None, render=True, **params):
    smap = smap or SMAP
    for target in TARGET_LIST:
        if target >= SHOTS[-1][1]:
            print(f"\n=== {target:g}s: skipped, the video is only {SHOTS[-1][1]:.1f}s long")
            continue
        for mode in MODES:
            pieces, stats = select_units(UNITS, smap, target, mode, **params)
            RESULTS[(target, mode)] = (pieces, stats)
            print(f"\n=== {target:g}s {mode}: {stats['total']}s, units {' '.join(stats['units'])} {stats['notes']}")
            for p in pieces:
                where = f"{p.video_start:6.2f}s" + ("" if abs(p.video_start - p.audio_start) < 1e-6 else f" (audio {p.audio_start:.2f}s)")
                print(f"   {p.length:5.2f}s @ {where}  {p.units:<10} {p.note}")
            if render and pieces:
                out = f"/content/cut_{target:g}s_{mode}.mp4"
                render_preview(VIDEO, pieces, out)
                display(Video(out, embed=True, width=420))

run_cuts()

## 11. Tune the selector (no Gemini call)
Higher `cut_penalty` = fewer, longer pieces. Higher `imp_power` = importance matters more than length.

In [ ]:
run_cuts(render=False, cut_penalty=3.0, imp_power=1.5, dur_power=0.5, min_importance=1.0, use_hook=True, tolerance=0.1)

## 12. (Optional) Compare AGENTIC vs STATIC on the same units

In [ ]:
other = "STATIC" if PROCESSING == "AGENTIC" else "AGENTIC"
raw_other, secs_other = story_call(processing=other)
smap_other = parse_story(raw_other)
cmp = pd.DataFrame({"id": [u.id for u in UNITS], "text": [u.text[:50] for u in UNITS],
                    PROCESSING: [m["importance"] for m in SMAP["units"]],
                    other: [m["importance"] for m in smap_other["units"]]})
cmp["diff"] = cmp[PROCESSING] - cmp[other]
print(f"{PROCESSING}: {SECS:.1f}s   {other}: {secs_other:.1f}s   correlation: {cmp[PROCESSING].corr(cmp[other]):.2f}")
print("summaries:\n ", PROCESSING, "→", SMAP["summary"], "\n ", other, "→", smap_other["summary"])
display(cmp)

## 13. Save everything (to compare runs later)

In [ ]:
dump = {"video": VIDEO, "model": MODEL, "processing": PROCESSING, "seconds": SECS,
        "units": [asdict(u) for u in UNITS], "story_map": SMAP, "raw_answer": RAW_TEXT,
        "cuts": {f"{t:g}s_{m}": {"stats": s, "pieces": [asdict(p) for p in ps]} for (t, m), (ps, s) in RESULTS.items()}}
path = f"/content/story_map_{PROCESSING}_{int(time.time())}.json"
json.dump(dump, open(path, "w"), indent=1)
from google.colab import files
files.download(path)